# Unit 5 – Jaccard distance / dissimilarity
Three patients' pathology results are compared. Fever, Cough and Test-1..4 are **asymmetric binary** attributes (a positive result matters more than a negative one), so Y and P become 1 and N and A become 0. **Gender is a symmetric variable and is therefore left out of the Jaccard calculation.**

Jaccard dissimilarity = (f01 + f10) / (f01 + f10 + f11), where f11 counts attributes that are 1 in both people, and f01 / f10 count attributes where only one person has 1. f00 (both 0) is ignored.

In [1]:
import pandas as pd, numpy as np
from itertools import combinations
from scipy.spatial.distance import jaccard

raw = pd.DataFrame({
    'Name':   ['Jack', 'Mary', 'Jim'],
    'Gender': ['M', 'F', 'M'],
    'Fever':  ['Y', 'Y', 'Y'],
    'Cough':  ['N', 'N', 'P'],
    'Test-1': ['P', 'P', 'N'],
    'Test-2': ['N', 'A', 'N'],
    'Test-3': ['N', 'P', 'N'],
    'Test-4': ['A', 'N', 'A'],
}).set_index('Name')
raw

,Gender,Fever,Cough,Test-1,Test-2,Test-3,Test-4
Name,,,,,,,
Jack,M,Y,N,P,N,N,A
Mary,F,Y,N,P,A,P,N
Jim,M,Y,P,N,N,N,A


In [2]:
asym_cols = ['Fever', 'Cough', 'Test-1', 'Test-2', 'Test-3', 'Test-4']
binary = raw[asym_cols].replace({'Y': 1, 'P': 1, 'N': 0, 'A': 0}).astype(int)
binary

,Fever,Cough,Test-1,Test-2,Test-3,Test-4
Name,,,,,,
Jack,1,0,1,0,0,0
Mary,1,0,1,0,1,0
Jim,1,1,0,0,0,0


In [3]:
def jaccard_distance(a, b):
    f11 = int(((a == 1) & (b == 1)).sum())
    f10 = int(((a == 1) & (b == 0)).sum())
    f01 = int(((a == 0) & (b == 1)).sum())
    return f01, f10, f11, (f01 + f10) / (f01 + f10 + f11)

rows = []
for p, q in [('Jack', 'Mary'), ('Jack', 'Jim'), ('Jim', 'Mary')]:
    f01, f10, f11, d = jaccard_distance(binary.loc[p].values, binary.loc[q].values)
    rows.append({'pair': f'({p}, {q})', 'f11': f11, 'f10': f10, 'f01': f01,
                 'Jaccard distance': round(d, 2),
                 'scipy check': round(jaccard(binary.loc[p].values, binary.loc[q].values), 2),
                 'Jaccard similarity': round(1 - d, 2)})
result = pd.DataFrame(rows); result

,pair,f11,f10,f01,Jaccard distance,scipy check,Jaccard similarity
0,"(Jack, Mary)",2,0,1,0.33,0.33,0.67
1,"(Jack, Jim)",1,1,1,0.67,0.67,0.33
2,"(Jim, Mary)",1,1,2,0.75,0.75,0.25


In [4]:
expected = {'(Jack, Mary)': 0.33, '(Jack, Jim)': 0.67, '(Jim, Mary)': 0.75}
assert all(abs(result.set_index('pair')['Jaccard distance'][k] - v) < 0.005 for k, v in expected.items())
print('All three distances match the published solution (0.33, 0.67, 0.75).')

names = list(binary.index)
dm = pd.DataFrame(0.0, index=names, columns=names)
for a, b in combinations(names, 2):
    dm.loc[a, b] = dm.loc[b, a] = jaccard_distance(binary.loc[a].values, binary.loc[b].values)[3]
dm.round(2)

All three distances match the published solution (0.33, 0.67, 0.75).


,Jack,Mary,Jim
Jack,0.00,0.33,0.67
Mary,0.33,0.00,0.75
Jim,0.67,0.75,0.00


## What if Gender were included?
Treating Gender as a binary attribute inflates the distance between people of different gender even though being male or female carries no 'presence' information.

In [5]:
with_gender = binary.copy(); with_gender['Gender_F'] = (raw['Gender'] == 'F').astype(int)
for p, q in [('Jack', 'Mary'), ('Jack', 'Jim'), ('Jim', 'Mary')]:
    d = jaccard_distance(with_gender.loc[p].values, with_gender.loc[q].values)[3]
    print(f'({p}, {q}) with Gender_F included: {d:.2f}')

(Jack, Mary) with Gender_F included: 0.50
(Jack, Jim) with Gender_F included: 0.67
(Jim, Mary) with Gender_F included: 0.80


## Observations
* Results match the published solution: (Jack, Mary) = 0.33, (Jack, Jim) = 0.67, (Jim, Mary) = 0.75.
* Jack and Mary are the most similar pair (they share a fever and a positive Test-1, and Mary has one extra positive). Jim and Mary are the most dissimilar: they share only the fever, while each has positives the other lacks.
* Counting joint negatives (f00) would make everyone look similar because most results are negative or absent; Jaccard ignores them on purpose.
* Including gender as a binary attribute changes the distances (0.50, 0.67, 0.80) and mixes a symmetric variable into an asymmetric measure, which is why the hint excludes it.